# Processing Big Data - Data Ingestion
© Explore Data Science Academy

## Honour Code
I'm Nabhan, confirm - by submitting this document - that the solutions in this notebook are a result of my own work and that I abide by the [EDSA honour code](https://drive.google.com/file/d/1QDCjGZJ8-FmJE3bZdIQNwnJyQKPhHZBn/view?usp=sharing).
    Non-compliance with the honour code constitutes a material breach of contract.



## Context 

To work constructively with any dataset, one needs to create an ingestion profile to make sure that the data at the source can be readily consumed. For this section of the predict, as the Data Engineer in the team, you will be required to design and implement the ingestion process. For the purposes of the project the AWS cloud storage service, namely, the S3 bucket service will act as your data source. All the data required can be found [here](https://processing-big-data-predict-stocks-data.s3.eu-west-1.amazonaws.com/stocks.zip).

<div align="center" style="width: 600px; font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/data_engineering/transform/predict/DataIngestion.jpg"
     alt="Data Ingestion"
     style="float: center; padding-bottom=0.5em"
     width=40%/>
     <p><em>Figure 1. Data Ingestion</em></p>
</div>

Your manager, Gnissecorp Atadgib, knowing very well that you've recently completed your Data Engineering qualification, asks you to make use of Apache Spark for the ingestion as well as the rest of the project. His rationale being, that stock market data is generated every day and is quite time-sensitive and would require scalability when deploying to a production environment. 

## Dataset - US Nasdaq




<div align="center" style="width: 600px; font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/data_engineering/transform/predict/Nasdaq.png"
     alt="Nasdaq"
     style="float: center; padding-bottom=0.5em"
     width=50%/>
     <p><em>Figure 2. Nasdaq</em></p>
</div>

The data that you will be working with is a historical snapshot of market data taken from the Nasdaq electronic market. This dataset contains historical daily prices for all tickers currently trading on Nasdaq. The up-to-date list can be found on their [website](https://www.nasdaq.com/)


The provided data contains price data dating back from 02 January 1962 up until 01 April 2020. The data found in the S3 bucket has been stored in the following structure:

```
     stocks/<Year>/<Month>/<Day>/stocks.csv
```
Each CSV file for every trading day contains the following details:
- **Date** - specifies trading date
- **Open** - opening price
- **High** - maximum price during the day
- **Low** - minimum price during the day
- **Close** - close price adjusted for splits
- **Adj Close** - close price adjusted for both dividends and splits
- **Volume** - the number of shares that changed hands during a given day

## Basic initialisation
To get you started, let's import some basic Python libraries as well as Spark modules and functions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

Remember that we need a `SparkContext` and `SparkSession` to interface with Spark.
We will mostly be using the `SparkContext` to interact with RDDs and the `SparkSession` to interface with Python objects.

> ℹ️ **Instructions** ℹ️
>
>Initialise a new **Spark Context** and **Session** that you will use to interface with Spark.

In [2]:
sc = SparkContext.getOrCreate()
spark = SparkSession.builder \
    .appName("StockDataIngestion") \
    .getOrCreate()

sc

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/05 14:06:08 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/09/05 14:06:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/09/05 14:06:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=pyspark-shell>

## Investigate dataset schema
At this point, it is enough to read in a single file to ascertain the data structure. You will be required to use the information obtained from the small subset to create a data schema. This data schema will be used when reading the entire dataset using Spark.

> ℹ️ **Instructions** ℹ️
>
>Make use of Pandas to read in a single file and investigate the plausible data types to be used when creating a Spark data schema. 
>
>*You may use as many coding cells as necessary.*

In [3]:
# Note: this notebook originally reads a per-day file from the S3
# stocks/<year>/<month>/<day>/stocks.csv structure. Here we use a local
# stocks.csv that already contains one trading day (2020-01-02) for every
# ticker, plus an extra "stock" column identifying the ticker.

sample_pdf = pd.read_csv("stocks.csv")
print(sample_pdf.dtypes)
sample_pdf.head()

Date             str
Open         float64
High         float64
Low          float64
Close        float64
Adj Close    float64
Volume       float64
stock            str
dtype: object


,Date,Open,High,Low,Close,Adj Close,Volume,stock
0,2020-01-02,85.900002,86.349998,85.199997,85.949997,85.731819,1410500.0,A
1,2020-01-02,21.860001,21.860001,21.315001,21.420000,21.420000,3062500.0,AA
2,2020-01-02,1.350000,1.380000,1.350000,1.350000,1.350000,4300.0,AACG
3,2020-01-02,28.980000,29.299999,28.650000,29.090000,28.982893,6451100.0,AAL
4,2020-01-02,12.310000,13.170000,12.310000,13.170000,13.170000,1700.0,AAMC


## Read CSV files

When working with big data, it is often not tenable to keep processing an entire data batch when you are in the process of development - this can be quite time-consuming. If the data is uniform, it is sufficient to work with a smaller subset to create basic functionality. Your manager has identified the year **1962** to perform the initial testing for data ingestion. 

> ℹ️ **Instructions** ℹ️
>
>Read in the data for **1962** using a data schema that purely uses string data types. You will be required to convert to the appropriate data types at a later stage.
>
>*You may use as many coding cells as necessary.*

In [4]:
# Read the CSV using a schema where every column is a string.
# We'll convert to proper types later, once we've checked for nulls.
schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Open", StringType(), True),
    StructField("High", StringType(), True),
    StructField("Low", StringType(), True),
    StructField("Close", StringType(), True),
    StructField("Adj Close", StringType(), True),
    StructField("Volume", StringType(), True),
    StructField("stock", StringType(), True),
])

df = spark.read.csv("stocks.csv", header=True, schema=schema)
df.show(5)
df.printSchema()

+----------+------------------+------------------+------------------+------------------+------------------+---------+-----+
|      Date|              Open|              High|               Low|             Close|         Adj Close|   Volume|stock|
+----------+------------------+------------------+------------------+------------------+------------------+---------+-----+
|2020-01-02|  85.9000015258789|  86.3499984741211| 85.19999694824219| 85.94999694824219| 85.73181915283203|1410500.0|    A|
|2020-01-02| 21.86000061035156| 21.86000061035156|21.315000534057614|21.420000076293945|21.420000076293945|3062500.0|   AA|
|2020-01-02| 1.350000023841858|1.3799999952316284| 1.350000023841858| 1.350000023841858| 1.350000023841858|   4300.0| AACG|
|2020-01-02|28.979999542236328| 29.29999923706055|28.649999618530273| 29.09000015258789|28.982892990112305|6451100.0|  AAL|
|2020-01-02|  12.3100004196167|13.170000076293945|  12.3100004196167|13.170000076293945|13.170000076293945|   1700.0| AAMC|
+-------

## Update column names
To make the data easier to work with, you will need to make a few changes:
1. Column headers should all be in lowercase; and
2. Whitespaces should be replaced with underscores.


> ℹ️ **Instructions** ℹ️
>
>Make sure that the column headers are all in lowercase and that any whitespaces are replaced with underscores.
>
>*You may use as many coding cells as necessary.*

In [5]:
# Lowercase all column headers and replace whitespace with underscores
new_columns = [c.lower().replace(" ", "_") for c in df.columns]
df = df.toDF(*new_columns)
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- open: string (nullable = true)
 |-- high: string (nullable = true)
 |-- low: string (nullable = true)
 |-- close: string (nullable = true)
 |-- adj_close: string (nullable = true)
 |-- volume: string (nullable = true)
 |-- stock: string (nullable = true)



## Null Values
Null values often represent missing pieces of data. It is always good to know where your null values lie - so you can quickly identify and remedy any issues stemming from these.

> ℹ️ **Instructions** ℹ️
>
>Write code to count the number of null values found in each column.
>
>*You may use as many coding cells as necessary.*

In [6]:
# Count null values per column (kept as a DataFrame so we can compare
# against the post-cast counts later)
pre_cast_nulls = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])
pre_cast_nulls.show()

+----+----+----+---+-----+---------+------+-----+
|date|open|high|low|close|adj_close|volume|stock|
+----+----+----+---+-----+---------+------+-----+
|   0|   0|   0|  0|    0|        0|     0|    0|
+----+----+----+---+-----+---------+------+-----+



## Data type conversion - The final data schema

Now that we have identified the number of missing values in the data set, we'll move on to convert our data schema to the required data types. 

> ℹ️ **Instructions** ℹ️
>
>Use typecasting to convert the string data types in your current data schema to more appropriate data types.
>
>*You may use as many coding cells as necessary.*

In [7]:
# Cast each string column to its appropriate data type.
# Note: "volume" is stored as a float-looking string (e.g. "1410500.0"),
# so it must go through DoubleType before LongType - a direct string -> bigint
# cast fails under Spark's ANSI mode.
df = (
    df.withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
      .withColumn("open", F.col("open").cast(DoubleType()))
      .withColumn("high", F.col("high").cast(DoubleType()))
      .withColumn("low", F.col("low").cast(DoubleType()))
      .withColumn("close", F.col("close").cast(DoubleType()))
      .withColumn("adj_close", F.col("adj_close").cast(DoubleType()))
      .withColumn("volume", F.col("volume").cast(DoubleType()).cast(LongType()))
)

df.printSchema()

root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- adj_close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- stock: string (nullable = true)



## Consolidate missing values
We have to check if the data type conversion above was done correctly.
If the casting was not successful, a null value gets inserted into the dataframe. You can thus check for successful conversion by determining if any null values are included in the resulting dataframe.


> ℹ️ **Instructions** ℹ️
>
>Write code to compare the number of invalid entries (nulls) pre-conversion and post-conversion.
>
>*You may use as many coding cells as necessary.*

In [8]:
# If a cast failed, Spark inserts a null - so a rise in null count here
# (versus pre_cast_nulls above) flags a row that didn't convert cleanly.
post_cast_nulls = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])

pre_counts = pre_cast_nulls.collect()[0].asDict()
post_counts = post_cast_nulls.collect()[0].asDict()

for col in df.columns:
    print(f"{col}: before={pre_counts.get(col, 0)}, after={post_counts[col]}")

date: before=0, after=0
open: before=0, after=0
high: before=0, after=0
low: before=0, after=0
close: before=0, after=0
adj_close: before=0, after=0
volume: before=0, after=0
stock: before=0, after=0


Here you should be able to see if any of your casts went wrong. 
Do not attempt to correct any missing values at this point. This will be dealt with in later sections of the predict.

## Generate parquet files
When writing in Spark, we typically use parquet format. This format allows parallel writing using Spark's optimisation while maintaining other useful things like metadata.

When writing, it is good to make sure that the data is sufficiently partitioned. 

Generally, data should be partitioned with one partition for every 200MB of data, but this also depends on the size of your cluster and executors. 


### Check the size of the dataframe before partitioning

In [9]:
from pyspark.serializers import PickleSerializer, AutoBatchedSerializer

In [10]:
rdd = df.rdd._reserialize(AutoBatchedSerializer(PickleSerializer()))
obj = rdd.ctx._jvm.org.apache.spark.mllib.api.python.SerDe.pythonToJava(rdd._jrdd, True)
size = sc._jvm.org.apache.spark.util.SizeEstimator.estimate(obj)
size_MB = size/1000000
partitions = max(int(size_MB/200), 2)
print(f'The dataframe is {size_MB} MB')

The dataframe is 4.471944 MB


### Write parquet files to the local directory
> ℹ️ **Instructions** ℹ️
>
> Use the **coalesce** function and the number of **partitions** derived above to write parquet files to your local directory 
>
>*You may use as many coding cells as necessary.*

In [11]:
# Write out as partitioned parquet files
df.coalesce(partitions).write.mode("overwrite").parquet("stocks_parquet/")